# IA (XMCO01): Atividade de Aprendizado de Máquina

Esta parte apresenta o trabalho, explora o dataset **Bank Marketing** e cria os dois conjuntos usados por todo o grupo:
`X_orig` (com `duration`), `X_mod` (sem `duration`) e `y` (alvo). Execute as células em ordem.

## 1. Objetivo e conceitos

| Tarefa | Tipo | Modelo | Avaliação |
|---|---|---|---|
| Classificação | Supervisionado (com "gabarito") | Random Forest e MLP | `classification_report` + matriz de confusão |
| Agrupamento | Não supervisionado (sem "gabarito") | K-Means | Índice de Silhueta |

Todos os modelos serão aplicados ao conjunto **original** e ao **modificado**, e os resultados serão comparados.

## 2. Dataset

Banco português que ligava para clientes oferecendo um **depósito a prazo**. Cada linha é um cliente.
O alvo `y` indica se ele **aceitou** (`yes`) ou não (`no`). Arquivo: `bank-full.csv` (45.211 linhas, 16 atributos + `y`).

- **Numéricos:** `age`, `balance`, `day`, `duration` (duração da ligação em segundos), `campaign`, `pdays`, `previous`
- **Categóricos:** `job`, `marital`, `education`, `default`, `housing`, `loan`, `contact`, `month`, `poutcome`

In [ ]:
import zipfile, urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42   # mesma semente para o grupo todo (resultados repetíveis)
pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (8, 4)

def localizar_ou_baixar(nome="bank-full.csv"):
    """Procura o CSV por perto; se não achar, baixa da UCI."""
    for pasta in [Path("."), Path("/content"), Path("dados")]:
        if (pasta / nome).exists():
            return pasta / nome
    try:
        Path("dados").mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/222/bank+marketing.zip",
                                   "dados/bank_marketing.zip")
        with zipfile.ZipFile("dados/bank_marketing.zip") as z: z.extractall("dados")
        with zipfile.ZipFile("dados/bank.zip") as z: z.extractall("dados")
        return Path("dados") / nome
    except Exception as e:
        raise FileNotFoundError(f"Download falhou ({e}). Baixe o zip em "
            "https://archive.ics.uci.edu/dataset/222/bank+marketing, extraia (há um bank.zip dentro) "
            "e coloque o bank-full.csv ao lado deste notebook.")

df = pd.read_csv(localizar_ou_baixar(), sep=";")   # separador é ';'
print(f"{df.shape[0]:,} linhas x {df.shape[1]} colunas")
df.head()

## 3. Análise exploratória
### 3.1 Tipos de dados e valores `unknown`

In [ ]:
alvo = "y"
cat_cols = df.drop(columns=alvo).select_dtypes(include="object").columns.tolist()
num_cols = df.drop(columns=alvo).select_dtypes(exclude="object").columns.tolist()
print("Numéricas  :", num_cols)
print("Categóricas:", cat_cols)
print("\nValores nulos no dataset:", int(df.isnull().sum().sum()))

# 'unknown' = informação não registrada (o pandas não o vê como nulo)
unk = (df[cat_cols] == "unknown").sum()
unk = unk[unk > 0].sort_values(ascending=False)
print("\nColunas com 'unknown':"); display(unk.to_frame("qtd").assign(pct=(unk / len(df) * 100).round(1)))

Não há nulos tradicionais. **Decisão:** manter `unknown` como uma categoria comum (também carrega informação e evita perder linhas).

### 3.2 Variável alvo e desbalanceamento

In [ ]:
y = df["y"].map({"no": 0, "yes": 1})   # yes = 1 é a classe de interesse
print(df["y"].value_counts(normalize=True).mul(100).round(1).astype(str) + " %")

# "Modelo burro" que sempre responde 'no'
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score
pred = DummyClassifier(strategy="most_frequent").fit(df, y).predict(df)
print(f"\nSempre responder 'no' -> acurácia: {accuracy_score(y, pred):.1%} | recall de 'yes': {recall_score(y, pred):.1%}")

**Conclusão:** o dataset é **desbalanceado** (~88% `no` / ~12% `yes`). Um modelo inútil já acerta ~88%, então a **acurácia engana**.
Nas análises, o foco será **recall, precision e F1 da classe `yes`** e a matriz de confusão.

### 3.3 Atributos numéricos e categóricos

In [ ]:
display(df[num_cols].describe().T[["mean", "std", "min", "50%", "max"]])

# Taxa de aceitação (% de 'yes') por categoria em alguns atributos
fig, eixos = plt.subplots(1, 3, figsize=(16, 4.5))
media = (df["y"] == "yes").mean() * 100
for ax, col in zip(eixos, ["job", "month", "poutcome"]):
    df.groupby(col)["y"].apply(lambda s: (s == "yes").mean() * 100).sort_values().plot(kind="barh", ax=ax, color="#dd8452")
    ax.axvline(media, color="black", ls="--"); ax.set_title(f"% de aceitação por '{col}'")
plt.tight_layout(); plt.show()

As escalas variam muito (`age` ~18-95; `balance` de negativos até >100 mil), por isso o Membro 2 **padronizará** os dados (MLP e K-Means são sensíveis à escala).
A linha tracejada é a média geral: barras à direita aceitam mais que a média, então esses atributos ajudam a prever `y`.

## 4. O atributo `duration` e por que removê-lo

In [ ]:
print("Duração média (s):", df.groupby("y")["duration"].mean().round(0).to_dict())

faixas = pd.cut(df["duration"], [-1, 60, 120, 180, 300, 600, 1000, df["duration"].max()],
                labels=["<1min", "1-2", "2-3", "3-5", "5-10", "10-17", ">17min"])
taxa = df.groupby(faixas, observed=True)["y"].apply(lambda s: (s == "yes").mean() * 100)
ax = taxa.plot(kind="bar", color="#dd8452", title="% de aceitação por duração da ligação")
ax.set_ylabel("% de aceitação"); plt.xticks(rotation=0); plt.show()

Quanto mais longa a ligação, maior a chance de aceitar. Parece ótimo, mas há uma armadilha:

> A duração **só é conhecida depois** da ligação, quando o resultado já é sabido. É como prever o jogo olhando o placar final (**data leakage**).

| Conjunto | Expectativa |
|---|---|
| **Original** (com `duration`) | Métricas altas, porém "infladas" |
| **Modificado** (sem `duration`) | Métricas menores, principalmente recall e F1 da classe `yes`, mas mais realistas |

## 5. Criando os dois conjuntos
`X` = características (entrada); `y` = alvo (resposta).

In [ ]:
X_orig = df.drop(columns="y")                  # 16 colunas (com duration)
X_mod  = X_orig.drop(columns="duration")       # 15 colunas (sem duration)

# Verificações de segurança
assert "duration" in X_orig and "duration" not in X_mod and "y" not in X_orig
assert X_orig.shape[1] == X_mod.shape[1] + 1 and len(X_orig) == len(X_mod) == len(y)

num_orig = num_cols
num_mod  = [c for c in num_cols if c != "duration"]
print(f"X_orig: {X_orig.shape} | X_mod: {X_mod.shape} | y: {y.shape}  -> verificações OK ✔")
print("Numéricas (modificado):", num_mod)